<a href="https://colab.research.google.com/github/aqrlouhanjoauhan/PakePlus-Android-v2.1.5/blob/main/youtube_subtitle_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 🚀 YouTube Knowledge Base Cloud Extraction Engine (Strict English Safe Guard) { display-mode: "form" }
#@markdown 💡 **Click the Run button (▶) on the left to start downloading subtitles!**

import os, re, shutil, json, subprocess, glob, time, random
from google.colab import output, files

TARGET_URL = ""

# ==============================================================
# 1. 弹窗引导用户粘贴 URL
# ==============================================================
js_prompt_code = """
(async () => {
    let clipboardText = "";
    try {
        const text = await navigator.clipboard.readText();
        if (text && (text.includes('youtube.com') || text.includes('youtu.be'))) {
            clipboardText = text;
        }
    } catch(e) {}

    const userEntered = prompt("🎯 Please paste your YouTube Channel or Video URL below (Ctrl+V / Cmd+V):", clipboardText);
    return { target_url: userEntered };
})();
"""

try:
    res = output.eval_js(js_prompt_code)
    if res and res.get('target_url') and res['target_url'].strip():
        TARGET_URL = res['target_url'].strip()
    else:
        TARGET_URL = ""
except Exception as e:
    TARGET_URL = ""

# 优雅退出
if not TARGET_URL:
    print("\n❌ Extraction Terminated: YouTube URL is empty or process was cancelled.")
    print("💡 Please click Run (▶) again and paste a valid YouTube link.")
    class StopExecution(Exception):
        def _render_traceback_(self):
            pass
    raise StopExecution

print(f"\n🎯 Target URL Confirmed: {TARGET_URL}")
print("⏳ Preparing cloud environment (takes ~10 seconds on first run)...")
os.system("pip install -q --upgrade youtube-transcript-api yt-dlp")
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound

base_dir = "./transcripts_temp"
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir, exist_ok=True)

# 纯净字幕清洗器
def clean_subtitle_text(text):
    if not text:
        return ""
    if text.strip().startswith("{") and ("replayChatItemAction" in text or "liveChat" in text or "clickTrackingParams" in text):
        return ""

    lines = text.splitlines()
    cleaned = []
    for line in lines:
        line = line.strip()
        if not line or "WEBVTT" in line or "Kind:" in line or "Language:" in line or "-->" in line:
            continue
        if re.match(r'^\d+$', line):
            continue
        line = re.sub(r'<[^>]+>', '', line)
        if not cleaned or cleaned[-1] != line:
            cleaned.append(line)
    return " ".join(cleaned)

# 🌟 强效英文字符合法性校验（拦截非英文乱码）
def is_valid_english_text(text):
    if not text or len(text.strip()) < 20:
        return False
    # 统计英文字母及常用标点的比例
    latin_chars = len(re.findall(r'[a-zA-Z0-9\s\.,\?!\'\"]', text))
    ratio = latin_chars / len(text)
    # 如果拉美/拉丁字符占比小于 70%，说明混入了阿布哈兹语/俄语/日语等非英文文本
    return ratio > 0.7

# 🌟 严格英文字幕提取器（绝不下载非英文字符）
def fetch_strict_english_transcript(vid):
    extracted = ""
    en_langs = ['en', 'en-US', 'en-GB', 'en-CA', 'en-AU', 'en-IN']

    # --- 策略 1: API 模式 ---
    try:
        t_list = YouTubeTranscriptApi.list_transcripts(vid)
        t_obj = None

        # 1.1 获取原生英文
        try:
            t_obj = t_list.find_transcript(en_langs)
        except Exception:
            pass

        # 1.2 强制尝试翻译为英文
        if not t_obj:
            try:
                first_t = next(iter(t_list))
                if first_t.is_translatable:
                    t_obj = first_t.translate('en')
            except Exception:
                pass

        if t_obj:
            fetched = t_obj.fetch()
            raw_text = " ".join([item.get('text', '') for item in fetched])
            extracted = clean_subtitle_text(raw_text)

            # 🌟 增加严格英文字符比重校验
            if extracted.strip() and is_valid_english_text(extracted):
                return extracted.strip(), False
            else:
                extracted = "" # 校验失败，直接丢弃

    except (TranscriptsDisabled, NoTranscriptFound):
        return "", True
    except Exception:
        pass

    # --- 策略 2: yt-dlp 严格英文模式 ---
    try:
        v_url = f"https://www.youtube.com/watch?v={vid}"
        temp_vtt_prefix = f"/tmp/sub_{vid}"

        dl_cmd = [
            "yt-dlp", "--skip-download",
            "--write-sub", "--write-auto-sub",
            "--sub-langs", "en.*,en",                  # 🌟 严格锁定英文
            "--sub-format", "vtt/srt/best",
            "--no-write-comments",
            "--user-agent", "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            "--extractor-args", "youtube:player_client=android,web",
            "--output", f"{temp_vtt_prefix}.%(ext)s", v_url
        ]
        subprocess.run(dl_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        downloaded_files = glob.glob(f"{temp_vtt_prefix}*")
        valid_sub_files = [f for f in downloaded_files if not f.endswith('.json')]

        en_files = [f for f in valid_sub_files if 'en' in os.path.basename(f).lower()]
        if en_files:
            selected_file = en_files[0]
            with open(selected_file, "r", encoding="utf-8", errors="ignore") as vf:
                raw_vtt = vf.read()
                extracted = clean_subtitle_text(raw_vtt)

        for df in downloaded_files:
            try: os.remove(df)
            except: pass

        if extracted.strip() and is_valid_english_text(extracted):
            return extracted.strip(), False
    except Exception:
        pass

    return "", False

# ==============================================================
# 2. 提取 Video ID / 频道列表
# ==============================================================
video_list = []
channel_title = "youtube_subtitles"

def extract_single_video_id(url):
    patterns = [
        r'(?:v=|\/)([0-9A-Za-z_-]{11}).*',
        r'youtu\.be\/([0-9A-Za-z_-]{11})',
        r'shorts\/([0-9A-Za-z_-]{11})'
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None

single_vid = extract_single_video_id(TARGET_URL)

if single_vid:
    print(f"\n🎬 Mode Detected: 【Single Video】(Extracted ID: {single_vid})")
    cmd = ["yt-dlp", "--dump-json", "--no-playlist", f"https://www.youtube.com/watch?v={single_vid}"]
    res = subprocess.run(cmd, capture_output=True, text=True)
    video_title = single_vid
    if res.returncode == 0:
        try:
            data = json.loads(res.stdout)
            video_title = data.get('title', single_vid)
        except Exception:
            pass

    video_list.append({'id': single_vid, 'title': video_title})
    safe_title = re.sub(r'[^\w\-_]', '_', video_title)[:30]
    channel_title = f"Video_{safe_title}"
else:
    print(f"\n📺 Mode Detected: 【Channel / Playlist Full Batch Download】")
    fetch_url = TARGET_URL.rstrip('/')
    if not any(fetch_url.endswith(sub) for sub in ['/videos', '/shorts', '/playlists', '/streams']):
        fetch_url += '/videos'

    cmd = ["yt-dlp", "--flat-playlist", "--dump-single-json", fetch_url]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        data = json.loads(res.stdout)
        channel_title = re.sub(r'[^\w\-_]', '_', data.get('title', 'Channel'))
        for entry in data.get('entries', []):
            if entry.get('id') and entry.get('_type') != 'playlist':
                video_list.append({'id': entry['id'], 'title': entry.get('title', entry['id'])})

        total_found = len(video_list)
        print(f"✅ Successfully scanned ALL {total_found} videos!")

# ==============================================================
# 3. 两轮严格英文批量提取
# ==============================================================
BATCH_SIZE = 50

if not video_list:
    print("❌ Extraction Failed: Could not recognize video info. Please check the URL.")
else:
    print("\n📝 Step 1/2: First Pass Extraction (Strict English Mode)...")
    success_count = 0
    failed_queue = []
    created_zips = []

    for idx, item in enumerate(video_list, 1):
        vid = item['id']
        title = item['title']
        display_title = title[:25] + "..." if len(title) > 25 else title

        text, is_no_sub = fetch_strict_english_transcript(vid)

        batch_num = ((idx - 1) // BATCH_SIZE) + 1
        batch_dir = os.path.join(base_dir, f"part_{batch_num}")
        os.makedirs(batch_dir, exist_ok=True)

        safe_title = re.sub(r'[^\w\-_]', '_', title)[:40]
        filepath = os.path.join(batch_dir, f"{idx:03d}_[{vid}]_{safe_title}.txt")

        if text:
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(text)
            success_count += 1
            print(f"  └─ [{idx}/{len(video_list)}] ✅ Success (English): {display_title}")
        elif is_no_sub:
            print(f"  └─ [{idx}/{len(video_list)}] ⚠️ Skipped (No English CC available): {display_title}")
        else:
            print(f"  └─ [{idx}/{len(video_list)}] ⏳ Busy/Blocked, queued for round 2: {display_title}")
            failed_queue.append({'idx': idx, 'vid': vid, 'title': title, 'filepath': filepath, 'display_title': display_title})

        time.sleep(random.uniform(0.3, 0.6))

    if failed_queue:
        print(f"\n🔄 Step 2/2: Retrying {len(failed_queue)} deferred video(s) after IP cooldown...")
        time.sleep(3.0)

        for retry_item in failed_queue:
            idx = retry_item['idx']
            vid = retry_item['vid']
            filepath = retry_item['filepath']
            display_title = retry_item['display_title']

            print(f"  └─ [{idx}/{len(video_list)}] 🔄 Re-trying now: {display_title}")

            text = ""
            for retry_attempt in range(1, 3):
                text, _ = fetch_strict_english_transcript(vid)
                if text:
                    break
                time.sleep(2.0)

            if text:
                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(text)
                success_count += 1
                print(f"  └─ [{idx}/{len(video_list)}] ✅ Recovered & Success (English): {display_title}")
            else:
                print(f"  └─ [{idx}/{len(video_list)}] ❌ Final Skipped (No valid English transcript): {display_title}")

    if success_count > 0:
        print(f"\n📦 Packing finished! Successfully fetched {success_count}/{len(video_list)} English transcripts.")
        part_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]
        part_folders.sort()

        for folder in part_folders:
            folder_path = os.path.join(base_dir, folder)
            if os.listdir(folder_path):
                zip_name = f"{channel_title}_en_subtitles" if len(part_folders) == 1 else f"{channel_title}_en_subtitles_{folder}"
                zip_filepath = shutil.make_archive(zip_name, 'zip', folder_path)
                created_zips.append(zip_filepath)

        print(f"🚀 Triggering download for {len(created_zips)} ZIP file(s)...")
        for zip_file in created_zips:
            files.download(zip_file)
    else:
        print("\n❌ Extraction Failed: Selected video(s) contain no valid English transcripts.")